# Thermal iteration engineering validation

This diagnostic validates the current repository implementation of `solve_iterative_thermal_state` without changing production code. It uses deterministic dry-air inputs, SI units, the real bare-tube geometry, the real property provider, direct correlation reconstruction, rating integration, and a point-of-use sentinel patch. Checks accumulate so one failed diagnostic does not hide later results.

Scope: dry, single-phase, bare round tubes; no condensation, phase change, fins, segmentation, or external benchmark. The current crossflow convention remains outside/inter-tube mixed and inside/tube-side unmixed; this notebook does not add or test public mixed/unmixed variants.


## 1. Environment and imports


In [1]:
from pathlib import Path
import dataclasses, inspect, math, platform, sys

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "core").is_dir():
            return candidate
    raise RuntimeError("Could not locate repository root from current working directory")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import core
import core.heat_transfer.thermal_iteration as thermal_iteration_module
import core.heat_transfer.internal_flow as internal_flow_module
import core.heat_transfer.outside_flow as outside_flow_module
import core.models.rating as rating_module
from core.geometry.tube import BareTube
from core.geometry.bundle import TubeBundle
from core.models.bare_tube import BareTubeHeatExchanger
from core.models.heat_balance import BalanceSideSpec
from core.properties.dry_air import DryAirPropertyProvider
from core.properties.adapters import to_internal_fluid_props
from core.properties.common import FluidTransportProperties
from core.heat_transfer.internal_flow import (
    gas_wall_temperature_correction, internal_length_correction,
    heat_transfer_coefficient_internal_diagnostics, nusselt_gnielinski,
)
from core.heat_transfer.outside_flow import prandtl_number as outside_prandtl_number
from core.heat_transfer.thermal_iteration import (
    solve_iterative_thermal_state, estimate_wall_temperature_envelope,
    IterativeThermalState, ThermalIterationDiagnostics,
)
from core.heat_transfer.outside_pressure_drop import GaddisGnielinskiEulerProvider

print("KalKalori version:", core.__version__)
print("Python version:", platform.python_version())
print("Repository root:", REPO_ROOT)
print("Solver module path:", inspect.getsourcefile(solve_iterative_thermal_state))
print("Solver signature:", inspect.signature(solve_iterative_thermal_state))
print("IterativeThermalState signature:", inspect.signature(IterativeThermalState))
print("Note: current API exports nusselt_gnielinski; no symbol named nusselt_internal_gnielinski exists in this repository.")


KalKalori version: 0.6.0
Python version: 3.11.9
Repository root: C:\Users\pawel\GitHub\kalkalori
Solver module path: C:\Users\pawel\GitHub\kalkalori\core\heat_transfer\thermal_iteration.py
Solver signature: (hx: "'BareTubeHeatExchanger'", *, m_dot_inside: 'float', m_dot_outside: 'float', inside_provider: 'PropertyProvider', outside_provider: 'PropertyProvider', T_in_inside: 'float', T_in_outside: 'float', p_inside: 'float', p_outside: 'float', flow_arrangement: 'str | None' = None, euler_provider: 'str' = 'zukauskas', max_iterations: 'int' = 25, wall_temperature_tolerance_K: 'float' = 0.05, relative_alfa_tolerance: 'float' = 0.001, relaxation_factor: 'float' = 0.5) -> 'IterativeThermalState'
IterativeThermalState signature: (inside_bulk_temperature: 'float', outside_bulk_temperature: 'float', inside_wall_temperature: 'float', outside_wall_temperature: 'float', inside_bulk_props: 'FluidTransportProperties', inside_wall_props: 'FluidTransportProperties | None', outside_bulk_props: 'Fluid

## 2. Validation framework and tolerances


In [2]:
checks = []
ABS_TOL_TEMPERATURE = 1e-3
REL_TOL_COEFFICIENT = 1e-6
REL_TOL_UA = 1e-6
REL_TOL_PROPERTIES = 1e-6
# Returned wall temperatures are the last relaxed iterate, while final coefficients are
# evaluated once more. The solver's configured 0.05 K convergence tolerance therefore
# justifies a 0.10 K resistance-reconstruction tolerance (measured errors are reported).
WALL_RECONSTRUCTION_TOL_K = 0.10

def record_check(name, condition, calculated=None, expected=None, notes="", status=None):
    checks.append({
        "Check": name,
        "Status": status or ("PASS" if bool(condition) else "FAIL"),
        "Calculated value": calculated,
        "Expected condition": expected,
        "Notes": notes,
    })

def print_table(rows, columns=None):
    rows = list(rows)
    if not rows:
        print("(none)"); return
    columns = columns or list(rows[0])
    values = [[str(row.get(c, "")) for c in columns] for row in rows]
    widths = [max(len(c), *(len(row[i]) for row in values)) for i, c in enumerate(columns)]
    print(" | ".join(c.ljust(widths[i]) for i, c in enumerate(columns)))
    print("-+-".join("-" * w for w in widths))
    for row in values:
        print(" | ".join(row[i].ljust(widths[i]) for i in range(len(columns))))

def warning_rows(state):
    return [{"code": w.code, "severity": w.severity, "source": w.source, "message": w.message} for w in state.warnings]

def endpoint_envelope_for_state(state, T_in_inside, T_in_outside):
    return estimate_wall_temperature_envelope(
        hx, m_dot_inside=M_DOT_INSIDE, m_dot_outside=M_DOT_OUTSIDE,
        inside_provider=inside_provider, outside_provider=outside_provider,
        inside_inlet_temperature=T_in_inside,
        inside_outlet_temperature=2.0 * state.inside_bulk_temperature - T_in_inside,
        outside_inlet_temperature=T_in_outside,
        outside_outlet_temperature=2.0 * state.outside_bulk_temperature - T_in_outside,
        p_inside=P, p_outside=P, max_iterations=MAX_ITERATIONS,
        wall_temperature_tolerance_K=WALL_TOL_K,
    )

def wall_temperature_summary_rows(state, envelope):
    return [
        {"surface": "inside", "mean [K]": state.inside_wall_temperature,
         "estimated min [K]": envelope.inside_min, "estimated max [K]": envelope.inside_max},
        {"surface": "outside", "mean [K]": state.outside_wall_temperature,
         "estimated min [K]": envelope.outside_min, "estimated max [K]": envelope.outside_max},
    ]


## 3. Inspect the actual calculation path


In [3]:
definition_module = inspect.getmodule(solve_iterative_thermal_state)
consumer_source = inspect.getsource(rating_module.run_rating)
patch_symbol = "core.models.rating.solve_iterative_thermal_state"
print("Defined in:", definition_module.__name__)
print("Definition file:", inspect.getsourcefile(solve_iterative_thermal_state))
print("Rating consumer:", rating_module.run_rating.__module__)
print("Point-of-use patch symbol:", patch_symbol)
print("Consumer binds same function object:", rating_module.solve_iterative_thermal_state is solve_iterative_thermal_state)
print("Call visible in run_rating source:", "solve_iterative_thermal_state(" in consumer_source)
record_check("Rating imports iterative solver at point of use", rating_module.solve_iterative_thermal_state is solve_iterative_thermal_state, patch_symbol, "same function object")


Defined in: core.heat_transfer.thermal_iteration
Definition file: C:\Users\pawel\GitHub\kalkalori\core\heat_transfer\thermal_iteration.py
Rating consumer: core.models.rating
Point-of-use patch symbol: core.models.rating.solve_iterative_thermal_state
Consumer binds same function object: True
Call visible in run_rating source: True


## 4. Deterministic baseline exchanger


In [4]:
P = 101_325.0
M_DOT_INSIDE = 10.0
M_DOT_OUTSIDE = 8.0
T_COLD_IN = 280.0
T_HOT_IN = 1100.0
MAX_ITERATIONS = 25
WALL_TOL_K = 0.05

tube = BareTube(D_i=0.022, D_o=0.025, length_total=2.8, length_effective=2.8, wall_k=16.0)
bundle = TubeBundle(
    tube=tube, n_rows=36, n_tubes_per_row=56,
    pitch_transverse=0.035, pitch_longitudinal=0.035,
    layout="staggered", n_passes_tube=1, flow_arrangement="counterflow",
)
hx = BareTubeHeatExchanger(bundle)
# Require the installed real CoolProp backend: no fallback or network access.
inside_provider = DryAirPropertyProvider(prefer_coolprop=True, allow_fallback=False)
outside_provider = DryAirPropertyProvider(prefer_coolprop=True, allow_fallback=False)

inputs = {
    "inside inlet / outside-hot case [K]": T_COLD_IN,
    "outside inlet / outside-hot case [K]": T_HOT_IN,
    "inside mass flow [kg/s]": M_DOT_INSIDE,
    "outside mass flow [kg/s]": M_DOT_OUTSIDE,
    "pressure, both sides [Pa]": P,
    "D_i [m]": tube.D_i, "D_o [m]": tube.D_o,
    "length_effective per flow path [m]": tube.length_effective,
    "number of tubes": bundle.n_tubes_total,
    "inside area [m2]": bundle.total_inner_area,
    "outside area [m2]": bundle.total_outer_area,
    "wall conductivity [W/(m K)]": tube.wall_k,
    "cylindrical wall resistance [K/W]": hx.tube_wall_resistance(),
    "flow arrangement": bundle.flow_arrangement,
}
print_table([{"Input": k, "Value": v} for k, v in inputs.items()])
record_check("Positive exchanger areas", bundle.total_inner_area > 0 and bundle.total_outer_area > 0, (bundle.total_inner_area, bundle.total_outer_area), "A_i > 0 and A_o > 0")
record_check("Non-zero cylindrical wall resistance", hx.tube_wall_resistance() > 0, hx.tube_wall_resistance(), "R_wall > 0")


Input                                | Value                 
-------------------------------------+-----------------------
inside inlet / outside-hot case [K]  | 280.0                 
outside inlet / outside-hot case [K] | 1100.0                
inside mass flow [kg/s]              | 10.0                  
outside mass flow [kg/s]             | 8.0                   
pressure, both sides [Pa]            | 101325.0              
D_i [m]                              | 0.022                 
D_o [m]                              | 0.025                 
length_effective per flow path [m]   | 2.8                   
number of tubes                      | 2016                  
inside area [m2]                     | 390.1405686416405     
outside area [m2]                    | 443.34155527459154    
wall conductivity [W/(m K)]          | 16.0                  
cylindrical wall resistance [K/W]    | 2.2526609631763815e-07
flow arrangement                     | counterflow           


## 5. Direct call: outside hot, inside cold


In [5]:
outside_hot = solve_iterative_thermal_state(
    hx, m_dot_inside=M_DOT_INSIDE, m_dot_outside=M_DOT_OUTSIDE,
    inside_provider=inside_provider, outside_provider=outside_provider,
    T_in_inside=T_COLD_IN, T_in_outside=T_HOT_IN,
    p_inside=P, p_outside=P, max_iterations=MAX_ITERATIONS,
    wall_temperature_tolerance_K=WALL_TOL_K,
)
outside_hot_envelope = endpoint_envelope_for_state(outside_hot, T_COLD_IN, T_HOT_IN)

def state_subset(s):
    d = s.diagnostics
    return {
        "iterations": s.iterations, "converged": s.converged, "residual [K]": s.residual,
        "inside bulk T [K]": s.inside_bulk_temperature, "inside wall T [K]": s.inside_wall_temperature,
        "outside wall T [K]": s.outside_wall_temperature, "outside bulk T [K]": s.outside_bulk_temperature,
        "inside bulk props": s.inside_bulk_props, "inside wall props": s.inside_wall_props,
        "outside bulk props": s.outside_bulk_props, "outside wall props": s.outside_wall_props,
        "inside Nu base": d.inside_Nu_base, "inside length correction": d.inside_length_correction,
        "inside wall-T correction": d.inside_wall_temperature_correction,
        "inside combined correction": d.inside_combined_correction,
        "inside Nu corrected": d.inside_Nu_corrected, "inside alfa base": d.inside_alfa_base,
        "inside alfa corrected": d.inside_alfa_corrected,
        "outside Nu base": d.outside_Nu_base, "outside wall correction": d.outside_wall_property_correction,
        "outside Nu corrected": d.outside_Nu_corrected, "outside alfa corrected": s.alfa_o,
        "U [W/(m2 K)]": s.U, "UA [W/K]": s.UA,
        "warnings": [w.code for w in s.warnings],
    }
print_table([{"Quantity": k, "Value": v} for k, v in state_subset(outside_hot).items()])
print("Wall-temperature result summary (0D endpoint envelope):")
print_table(wall_temperature_summary_rows(outside_hot, outside_hot_envelope))
record_check("Direct iterative solver executed", isinstance(outside_hot, IterativeThermalState), type(outside_hot).__name__, "IterativeThermalState returned")
record_check("Normal outside-hot case converged", outside_hot.converged, (outside_hot.iterations, outside_hot.residual), "converged=True")


Quantity                   | Value                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

## 6. Physical wall-temperature ordering


In [6]:
s = outside_hot; tol = ABS_TOL_TEMPERATURE
physical = [
    ("Outside wall below outside bulk", s.outside_wall_temperature < s.outside_bulk_temperature + tol, s.outside_bulk_temperature - s.outside_wall_temperature, "> -tol K"),
    ("Inside wall above inside bulk", s.inside_wall_temperature > s.inside_bulk_temperature - tol, s.inside_wall_temperature - s.inside_bulk_temperature, "> -tol K"),
    ("Outside wall not colder than inside wall", s.outside_wall_temperature >= s.inside_wall_temperature - tol, s.outside_wall_temperature - s.inside_wall_temperature, ">= -tol K"),
    ("Both wall temperatures finite", math.isfinite(s.inside_wall_temperature) and math.isfinite(s.outside_wall_temperature), (s.inside_wall_temperature, s.outside_wall_temperature), "both finite"),
    ("Both walls bracketed by bulk temperatures", all(s.inside_bulk_temperature-tol <= x <= s.outside_bulk_temperature+tol for x in (s.inside_wall_temperature,s.outside_wall_temperature)), (s.inside_wall_temperature,s.outside_wall_temperature), "inside bulk <= walls <= outside bulk"),
]
for name, ok, calc, expected in physical: record_check(name, ok, calc, expected)
print_table([{"Temperature": k, "K": v} for k,v in [("outside bulk",s.outside_bulk_temperature),("outside wall",s.outside_wall_temperature),("inside wall",s.inside_wall_temperature),("inside bulk",s.inside_bulk_temperature)]])
print_table(wall_temperature_summary_rows(s, outside_hot_envelope))


Temperature  | K                
-------------+------------------
outside bulk | 844.8029009771644
outside wall | 736.2562285859442
inside wall  | 735.2322801163884
inside bulk  | 499.8334875340787
surface | mean [K]          | estimated min [K] | estimated max [K]
--------+-------------------+-------------------+------------------
inside  | 735.2322801163884 | 496.3833786141928 | 976.795548044211 
outside | 736.2562285859442 | 497.148217454641  | 978.0594522823106


## 7. Reverse heat-flow direction


In [7]:
inside_hot = solve_iterative_thermal_state(
    hx, m_dot_inside=M_DOT_INSIDE, m_dot_outside=M_DOT_OUTSIDE,
    inside_provider=inside_provider, outside_provider=outside_provider,
    T_in_inside=T_HOT_IN, T_in_outside=T_COLD_IN,
    p_inside=P, p_outside=P, max_iterations=MAX_ITERATIONS,
    wall_temperature_tolerance_K=WALL_TOL_K,
)
inside_hot_envelope = endpoint_envelope_for_state(inside_hot, T_HOT_IN, T_COLD_IN)
s = inside_hot
print_table([{"Temperature": k, "K": v} for k,v in [("inside bulk",s.inside_bulk_temperature),("inside wall",s.inside_wall_temperature),("outside wall",s.outside_wall_temperature),("outside bulk",s.outside_bulk_temperature)]])
print_table(wall_temperature_summary_rows(s, inside_hot_envelope))
reverse_order = s.inside_bulk_temperature > s.inside_wall_temperature-tol and s.inside_wall_temperature >= s.outside_wall_temperature-tol and s.outside_wall_temperature > s.outside_bulk_temperature-tol
record_check("Reverse case converged", s.converged, (s.iterations,s.residual), "converged=True")
record_check("Reverse wall-temperature ordering", reverse_order, (s.inside_bulk_temperature,s.inside_wall_temperature,s.outside_wall_temperature,s.outside_bulk_temperature), "Tbi > Twi >= Two > Tbo")
record_check("Signed heat-flow convention reverses", s.UA*(s.inside_bulk_temperature-s.outside_bulk_temperature) > 0 and outside_hot.UA*(outside_hot.inside_bulk_temperature-outside_hot.outside_bulk_temperature) < 0, (s.UA*(s.inside_bulk_temperature-s.outside_bulk_temperature),outside_hot.UA*(outside_hot.inside_bulk_temperature-outside_hot.outside_bulk_temperature)), "positive inside-to-outside only when inside is hot")
record_check("Gas perspective selects heating/cooling branch", outside_hot.diagnostics.inside_wall_temperature_correction < 1 and math.isclose(inside_hot.diagnostics.inside_wall_temperature_correction,1.0,abs_tol=1e-12), (outside_hot.diagnostics.inside_wall_temperature_correction,inside_hot.diagnostics.inside_wall_temperature_correction), "heated gas < 1; cooled gas = 1")


Temperature  | K                
-------------+------------------
inside bulk  | 886.5863494245054
inside wall  | 701.8855235372882
outside wall | 700.7854759067453
outside bulk | 565.904140799295 
surface | mean [K]          | estimated min [K]  | estimated max [K]
--------+-------------------+--------------------+------------------
inside  | 701.8855235372882 | 462.47110694446906 | 951.1690257102057
outside | 700.7854759067453 | 461.30177496452    | 950.2429045939234


## 8. Property-evaluation temperatures


In [8]:
PROPERTY_NAMES = ("rho","mu","k","cp")
def compare_props(label, reported, provider, T, p):
    expected = provider.at(T=T,p=p)
    for name in PROPERTY_NAMES:
        a,b = getattr(reported,name),getattr(expected,name)
        record_check(f"{label} {name} at reported temperature", math.isclose(a,b,rel_tol=REL_TOL_PROPERTIES,abs_tol=0.0), a, b, f"evaluated at {T:.9g} K")
    pr_a = reported.cp*reported.mu/reported.k
    pr_b = expected.cp*expected.mu/expected.k
    record_check(f"{label} Pr at reported temperature", math.isclose(pr_a,pr_b,rel_tol=REL_TOL_PROPERTIES), pr_a, pr_b, f"derived at {T:.9g} K")

for direction, state in (("outside-hot",outside_hot),("inside-hot",inside_hot)):
    compare_props(f"{direction} inside bulk",state.inside_bulk_props,inside_provider,state.inside_bulk_temperature,P)
    compare_props(f"{direction} inside wall",state.inside_wall_props,inside_provider,state.inside_wall_temperature,P)
    compare_props(f"{direction} outside bulk",state.outside_bulk_props,outside_provider,state.outside_bulk_temperature,P)
    compare_props(f"{direction} outside wall",state.outside_wall_props,outside_provider,state.outside_wall_temperature,P)
print("Compared rho, mu, k, cp and derived Pr for four property sets in both directions.")


Compared rho, mu, k, cp and derived Pr for four property sets in both directions.


## 9. Internal wall-temperature and finite-length corrections


In [9]:
print("Implemented gas correction source:\n", inspect.getsource(gas_wall_temperature_correction))
print("Implemented length correction source:\n", inspect.getsource(internal_length_correction))

for label,state in (("heated inside gas",outside_hot),("cooled inside gas",inside_hot)):
    T_b,T_w = state.inside_bulk_temperature,state.inside_wall_temperature
    exponent = internal_flow_module._GAS_HEATING_EXPONENT if T_w > T_b else 0.0
    expected_wall = (T_b/T_w)**exponent
    actual = state.diagnostics.inside_wall_temperature_correction
    print(label,{"branch":"heating" if exponent else "cooling","T_bulk":T_b,"T_wall_inside":T_w,"T_bulk/T_wall":T_b/T_w,"exponent":exponent,"factor":actual})
    record_check(f"{label}: wall correction formula",math.isclose(actual,expected_wall,rel_tol=1e-12),actual,expected_wall,"absolute kelvins; inside wall; raw final factor")

ratio = outside_hot.inside_bulk_temperature/outside_hot.inside_wall_temperature
record_check("Representative heated-gas ratio in requested practical range",0.66 <= ratio <= 0.71,ratio,"0.66 <= T_bulk/T_wall <= 0.71")
expected_length = 1.0 + (tube.D_i/tube.length_effective)**(2.0/3.0)
raw_length,_ = internal_length_correction(tube.D_i,tube.length_effective)
actual_length = outside_hot.diagnostics.inside_length_correction
print({"D_i":tube.D_i,"L_heated":tube.length_effective,"D_i/L_heated":tube.D_i/tube.length_effective,"raw_length_correction":raw_length,"total_parallel_tube_length":tube.length_effective*bundle.n_tubes_total,"multi_pass_hydraulic_length":bundle.internal_length_total})
record_check("Finite heated length is valid",math.isfinite(tube.length_effective) and tube.length_effective > 0,tube.length_effective,"finite and positive")
record_check("Heated length is one active path",tube.length_effective == tube.length_effective and tube.length_effective != tube.length_effective*bundle.n_tubes_total,tube.length_effective,"tube.length_effective, not aggregate parallel length")
record_check("Finite-length formula reconstructed",math.isclose(actual_length,expected_length,rel_tol=1e-12),actual_length,expected_length)
record_check("Finite-length correction active",actual_length > 1.0+1e-6,actual_length,"> 1 and non-negligible")


Implemented gas correction source:
 def gas_wall_temperature_correction(
    T_bulk: float,
    T_wall: float,
) -> tuple[float, list[ModelWarning]]:
    """
    Variable-property (bulk-to-wall temperature) correction factor for
    turbulent internal *gas* flow, to be applied on top of the
    constant-property Gnielinski Nusselt number.

        Nu / Nu_cp = (T_bulk / T_wall)^n      (T in absolute units, K)

            n = 0     for T_wall <= T_bulk (gas cooling, or no bulk/wall
                      difference)
            n = 0.45  for T_wall > T_bulk (gas heating)

    "Heating"/"cooling" are defined from the gas (tube-side) perspective by
    comparing ``T_wall`` (inside tube-wall surface temperature) directly to
    ``T_bulk`` (tube-side bulk temperature) -- never inferred from generic
    hot-stream/cold-stream identity, so the result is correct regardless of
    which physical side of the exchanger happens to run hot overall.

    Reference:
        VDI Heat Atlas, Section G1

## 10. Correction composition and Nu-to-alfa consistency


In [10]:
for label,state in (("outside-hot",outside_hot),("inside-hot",inside_hot)):
    d=state.diagnostics
    expected_combined=d.inside_length_correction*d.inside_wall_temperature_correction
    expected_nu=d.inside_Nu_base*expected_combined
    expected_ai_base=d.inside_Nu_base*state.inside_bulk_props.k/tube.D_i
    expected_ai=d.inside_Nu_corrected*state.inside_bulk_props.k/tube.D_i
    record_check(f"{label}: corrections composed exactly once",math.isclose(d.inside_combined_correction,expected_combined,rel_tol=1e-12),d.inside_combined_correction,expected_combined)
    record_check(f"{label}: corrected Nu applies combined factor once",math.isclose(d.inside_Nu_corrected,expected_nu,rel_tol=1e-12),d.inside_Nu_corrected,expected_nu)
    record_check(f"{label}: base alfa from base Nu",math.isclose(d.inside_alfa_base,expected_ai_base,rel_tol=REL_TOL_COEFFICIENT),d.inside_alfa_base,expected_ai_base,"bulk k / D_i")
    record_check(f"{label}: final alfa_i from corrected Nu",math.isclose(state.alfa_i,expected_ai,rel_tol=REL_TOL_COEFFICIENT),state.alfa_i,expected_ai,"bulk k / D_i")
    record_check(f"{label}: returned alfa equals diagnostic corrected alfa",math.isclose(state.alfa_i,d.inside_alfa_corrected,rel_tol=REL_TOL_COEFFICIENT),state.alfa_i,d.inside_alfa_corrected)
    print(label,{"alfa_base":d.inside_alfa_base,"alfa_wall_only":d.inside_alfa_base*d.inside_wall_temperature_correction,"alfa_fully_corrected":expected_ai,"alfa_returned":state.alfa_i})


outside-hot {'alfa_base': 56.64278129871348, 'alfa_wall_only': 47.612931601721456, 'alfa_fully_corrected': 49.49470802138283, 'alfa_returned': 49.49470802138283}
inside-hot {'alfa_base': 64.88211554494904, 'alfa_wall_only': 64.88211554494904, 'alfa_fully_corrected': 67.44641123065759, 'alfa_returned': 67.44641123065759}


## 11. Outside wall-property correction


In [11]:
for label,state in (("outside-hot",outside_hot),("inside-hot",inside_hot)):
    bulk=state.outside_bulk_props; wall=state.outside_wall_props
    Pr=outside_prandtl_number(bulk.cp,bulk.mu,bulk.k)
    Pr_s=outside_prandtl_number(wall.cp,wall.mu,wall.k)
    expected=(Pr/Pr_s)**0.25
    d=state.diagnostics
    record_check(f"{label}: outside correction uses wall Pr_s",math.isclose(d.outside_wall_property_correction,expected,rel_tol=1e-12),d.outside_wall_property_correction,expected,f"Pr={Pr:.9g}, Pr_s at T_wall_outside={Pr_s:.9g}")
    record_check(f"{label}: outside correction applied once",math.isclose(d.outside_Nu_corrected,d.outside_Nu_base*expected,rel_tol=1e-12),d.outside_Nu_corrected,d.outside_Nu_base*expected)
    wrong_bulk=(Pr/Pr)**0.25
    print(label,{"T_wall_outside":state.outside_wall_temperature,"T_outside_bulk":state.outside_bulk_temperature,"Pr":Pr,"Pr_s":Pr_s,"factor":d.outside_wall_property_correction,"factor_if_bulk_reused":wrong_bulk})


outside-hot {'T_wall_outside': 736.2562285859442, 'T_outside_bulk': 844.8029009771644, 'Pr': 0.7203264910299799, 'Pr_s': 0.712517223221571, 'factor': 1.0027288375673262, 'factor_if_bulk_reused': 1.0}
inside-hot {'T_wall_outside': 700.7854759067453, 'T_outside_bulk': 565.904140799295, 'Pr': 0.7010373250746347, 'Pr_s': 0.709894004534598, 'factor': 0.9968662857606584, 'factor_if_bulk_reused': 1.0}


## 12. Wall-temperature and UA reconstruction from final resistances


In [12]:
for label,state in (("outside-hot",outside_hot),("inside-hot",inside_hot)):
    A_i,A_o=bundle.total_inner_area,bundle.total_outer_area
    R_i=1.0/(state.alfa_i*A_i); R_w=hx.tube_wall_resistance(); R_o=1.0/(state.alfa_o*A_o)
    q_signed=state.UA*(state.inside_bulk_temperature-state.outside_bulk_temperature)
    Twi=state.inside_bulk_temperature-q_signed*R_i
    Two=state.outside_bulk_temperature+q_signed*R_o
    err_i=abs(Twi-state.inside_wall_temperature); err_o=abs(Two-state.outside_wall_temperature)
    UA_rec=1.0/(R_i+R_w+R_o); U_rec=UA_rec/A_o
    print(label,{"q_inside_to_outside":q_signed,"R_inside":R_i,"R_wall":R_w,"R_outside":R_o,"Twi_error_K":err_i,"Two_error_K":err_o,"UA_reconstructed":UA_rec})
    record_check(f"{label}: inside wall reconstructed",err_i <= WALL_RECONSTRUCTION_TOL_K,err_i,f"<= {WALL_RECONSTRUCTION_TOL_K} K","Tolerance reflects 0.05 K relaxed-iterate convergence threshold")
    record_check(f"{label}: outside wall reconstructed",err_o <= WALL_RECONSTRUCTION_TOL_K,err_o,f"<= {WALL_RECONSTRUCTION_TOL_K} K","Tolerance reflects 0.05 K relaxed-iterate convergence threshold")
    record_check(f"{label}: UA reconstructed",math.isclose(UA_rec,state.UA,rel_tol=REL_TOL_UA),UA_rec,state.UA)
    record_check(f"{label}: U uses outside-area basis",math.isclose(U_rec,state.U,rel_tol=REL_TOL_COEFFICIENT),U_rec,state.U,"U = UA/A_o")


outside-hot {'q_inside_to_outside': -4544408.760156944, 'R_inside': 5.1786924579968415e-05, 'R_wall': 2.2526609631763815e-07, 'R_outside': 2.389854527414196e-05, 'Twi_error_K': 0.05783885951427692, 'Two_error_K': 0.05808610759788735, 'UA_reconstructed': 13173.367211892531}
inside-hot {'q_inside_to_outside': 4860220.83358744, 'R_inside': 3.800318897094741e-05, 'R_wall': 2.2526609631763815e-07, 'R_outside': 2.7752540365921393e-05, 'Twi_error_K': 0.0030648921417650854, 'Two_error_K': 0.0021397639771976174, 'UA_reconstructed': 15155.879256362814}


## 13. Convergence inspection and correction sensitivities


In [13]:
for label,state in (("outside-hot",outside_hot),("inside-hot",inside_hot)):
    record_check(f"{label}: positive iteration count",state.iterations > 0,state.iterations,"> 0")
    record_check(f"{label}: iteration limit respected",state.iterations <= MAX_ITERATIONS,state.iterations,f"<= {MAX_ITERATIONS}")
    record_check(f"{label}: residual meets configured tolerance",(not state.converged) or state.residual < WALL_TOL_K,state.residual,f"< {WALL_TOL_K} K when converged")
    final_numbers=(state.inside_wall_temperature,state.outside_wall_temperature,state.alfa_i,state.alfa_o,state.U,state.UA,state.residual)
    record_check(f"{label}: all final values finite",all(math.isfinite(x) for x in final_numbers),final_numbers,"all finite")
record_check("Iteration history availability",True,"iterations/residual/converged only","history table if exposed","IterativeThermalState does not expose per-iteration history; production code was not changed.",status="NOT APPLICABLE")
print("No iteration history field is exposed; final convergence diagnostics:")
print_table([{"case":label,"iterations":s.iterations,"residual":s.residual,"converged":s.converged} for label,s in (("outside-hot",outside_hot),("inside-hot",inside_hot))])

bulk=outside_hot.inside_bulk_props
common=dict(m_dot=M_DOT_INSIDE,tube_inner_diameter=tube.D_i,flow_area=bundle.internal_flow_area_per_pass,props=to_internal_fluid_props(bulk))
base=heat_transfer_coefficient_internal_diagnostics(**common)
wall_only=heat_transfer_coefficient_internal_diagnostics(**common,T_bulk=outside_hot.inside_bulk_temperature,T_wall=outside_hot.inside_wall_temperature)
full=heat_transfer_coefficient_internal_diagnostics(**common,T_bulk=outside_hot.inside_bulk_temperature,T_wall=outside_hot.inside_wall_temperature,L_heated=tube.length_effective)
print_table([{"case":"base","Nu":base.Nu_corrected,"alfa":base.alfa_corrected},{"case":"wall only","Nu":wall_only.Nu_corrected,"alfa":wall_only.alfa_corrected},{"case":"wall + length","Nu":full.Nu_corrected,"alfa":full.alfa_corrected},{"case":"converged state","Nu":outside_hot.diagnostics.inside_Nu_corrected,"alfa":outside_hot.alfa_i}])
record_check("Heated-gas wall correction reduces Nu",wall_only.Nu_corrected < base.Nu_corrected,(base.Nu_corrected,wall_only.Nu_corrected),"wall-corrected < base for heating branch")
record_check("Full direct internal diagnostic matches converged state",math.isclose(full.alfa_corrected,outside_hot.alfa_i,rel_tol=REL_TOL_COEFFICIENT),full.alfa_corrected,outside_hot.alfa_i)

lengths=(0.10,tube.length_effective,1.0e9)
length_diags=[heat_transfer_coefficient_internal_diagnostics(**common,L_heated=L) for L in lengths]
print_table([{"L_heated [m]":L,"D/L":tube.D_i/L,"length factor":d.length_correction,"Nu":d.Nu_corrected} for L,d in zip(lengths,length_diags)])
record_check("Length correction decreases toward asymptote",length_diags[0].length_correction > length_diags[1].length_correction > length_diags[2].length_correction >= 1.0,[d.length_correction for d in length_diags],"short > baseline > very long >= 1")
record_check("Very-long length tends to one",math.isclose(length_diags[-1].length_correction,1.0,abs_tol=1e-6),length_diags[-1].length_correction,"approximately 1")


No iteration history field is exposed; final convergence diagnostics:
case        | iterations | residual            | converged
------------+------------+---------------------+----------
outside-hot | 18         | 0.04721200095968925 | True     
inside-hot  | 12         | 0.01601364469240707 | True     
case            | Nu                | alfa              
----------------+-------------------+-------------------
base            | 31.20485920471305 | 56.64278129871348 
wall only       | 26.23025905313545 | 47.612931601721456
wall + length   | 27.26694134316304 | 49.49470802138283 
converged state | 27.26694134316304 | 49.49470802138283 
L_heated [m] | D/L                    | length factor      | Nu                
-------------+------------------------+--------------------+-------------------
0.1          | 0.21999999999999997    | 1.3644308387191582 | 42.57687221679987 
2.8          | 0.007857142857142858   | 1.0395223809237855 | 32.43814953687481 
1000000000.0 | 2.199999999999999

## 14. Forced non-convergence and invalid inputs


In [14]:
forced = solve_iterative_thermal_state(
    hx,m_dot_inside=M_DOT_INSIDE,m_dot_outside=M_DOT_OUTSIDE,
    inside_provider=inside_provider,outside_provider=outside_provider,
    T_in_inside=T_COLD_IN,T_in_outside=T_HOT_IN,p_inside=P,p_outside=P,
    max_iterations=1,wall_temperature_tolerance_K=1e-300,
)
forced_codes={w.code for w in forced.warnings}
record_check("Forced non-convergence detected",not forced.converged and forced.iterations==1,(forced.converged,forced.iterations),"False, 1")
record_check("Forced non-convergence structured warning","thermal_iteration_not_converged" in forced_codes,sorted(forced_codes),"contains thermal_iteration_not_converged")
print("Forced case warnings:"); print_table(warning_rows(forced))

def expect_clear_failure(name,call):
    try:
        result=call()
    except (ValueError,TypeError) as exc:
        record_check(name,True,type(exc).__name__,"clear exception",str(exc)); return
    codes={getattr(w,"code","") for w in getattr(result,"warnings",())}
    record_check(name,any(codes),sorted(codes),"exception or structured critical warning")

base_kwargs=dict(hx=hx,m_dot_inside=M_DOT_INSIDE,m_dot_outside=M_DOT_OUTSIDE,inside_provider=inside_provider,outside_provider=outside_provider,T_in_inside=T_COLD_IN,T_in_outside=T_HOT_IN,p_inside=P,p_outside=P)
expect_clear_failure("Reject non-positive inside mass/capacity input",lambda:solve_iterative_thermal_state(**{**base_kwargs,"m_dot_inside":0.0}))
expect_clear_failure("Reject non-finite numerical input",lambda:solve_iterative_thermal_state(**{**base_kwargs,"T_in_inside":float("nan")}))
expect_clear_failure("Reject non-positive area geometry",lambda:BareTube(D_i=0.0,D_o=0.025,length_total=2.8,length_effective=2.8,wall_k=16.0))
expect_clear_failure("Reject non-positive wall conductivity",lambda:BareTube(D_i=0.022,D_o=0.025,length_total=2.8,length_effective=2.8,wall_k=0.0))
factor,bad_wall_warnings=gas_wall_temperature_correction(500.0,float("nan"))
record_check("Invalid directly accepted wall temperature warned",factor==1.0 and any(w.code=="tube_ht_gas_wall_correction_invalid_temperature" for w in bad_wall_warnings),(factor,[w.code for w in bad_wall_warnings]),"factor=1 with structured warning")


Forced case warnings:
code                                             | severity | source            | message                                                                                                                                            
-------------------------------------------------+----------+-------------------+----------------------------------------------------------------------------------------------------------------------------------------------------
outside_dp_zukauskas_staggered_sl_over_d_offgrid | info     | outside_dp        | outside_dp: current open Zukauskas provider is strongest near staggered SL/D = 1.25, 1.5, 2.0 and uses interpolation/clamping outside these points.
outside_dp_zukauskas_staggered_sl_over_d_offgrid | info     | outside_dp        | outside_dp: current open Zukauskas provider is strongest near staggered SL/D = 1.25, 1.5, 2.0 and uses interpolation/clamping outside these points.
thermal_iteration_not_converged                  | warning

## 15. Exchanger-level rating consumption


In [15]:
inside_spec=BalanceSideSpec(provider=inside_provider,p=P,m_dot=M_DOT_INSIDE,T_in=T_COLD_IN,T_out=None)
outside_spec=BalanceSideSpec(provider=outside_provider,p=P,m_dot=M_DOT_OUTSIDE,T_in=T_HOT_IN,T_out=None)
rating_result=hx.rate(inside_spec,outside_spec,effectiveness=0.30)
ts=rating_result.thermal_state
comparisons=(
    ("alfa_i",rating_result.alfa_i,ts.alfa_i),("alfa_o",rating_result.alfa_o,ts.alfa_o),
    ("U",rating_result.U_mean,ts.U),("UA",rating_result.UA_actual,ts.UA),
)
for name,top,nested in comparisons:
    record_check(f"Rating top-level {name} consumes iterative state",math.isclose(top,nested,rel_tol=REL_TOL_COEFFICIENT),top,nested)
record_check("Rating exposes iterative convergence status",ts.converged,(ts.iterations,ts.converged,ts.residual),"nested state converged")
record_check("Top-level wall temperatures/iteration fields",True,"available only in rating_result.thermal_state","direct top-level fields","HXRatingResult intentionally exposes these through its nested IterativeThermalState.",status="NOT APPLICABLE")
print_table([{"quantity":name,"top-level":top,"nested iterative state":nested} for name,top,nested in comparisons])
print({"inside_wall_temperature":ts.inside_wall_temperature,"outside_wall_temperature":ts.outside_wall_temperature,"iterations":ts.iterations,"converged":ts.converged})
print("Rating wall-temperature result summary (0D endpoint envelope):")
print_table(wall_temperature_summary_rows(ts, rating_result.wall_temperature_envelope))


quantity | top-level          | nested iterative state
---------+--------------------+-----------------------
alfa_i   | 49.49470802138283  | 49.49470802138283     
alfa_o   | 94.38219942542443  | 94.38219942542443     
U        | 29.71381106770686  | 29.71381106770686     
UA       | 13173.367211892531 | 13173.367211892531    
{'inside_wall_temperature': 735.2322801163884, 'outside_wall_temperature': 736.2562285859442, 'iterations': 18, 'converged': True}
Rating wall-temperature result summary (0D endpoint envelope):
surface | mean [K]          | estimated min [K] | estimated max [K]
--------+-------------------+-------------------+------------------
inside  | 735.2322801163884 | 710.7133602837991 | 935.2745348972218
outside | 736.2562285859442 | 712.0087201858892 | 936.9657193810052


## 16. Sentinel point-of-use call-path test


In [16]:
from unittest import mock
SENTINEL_AI,SENTINEL_AO,SENTINEL_U,SENTINEL_UA=123.456,234.567,16.789,3456.789
sentinel_props=FluidTransportProperties(rho=1.0,mu=2e-5,k=0.03,cp=1007.0)
sentinel_diag=ThermalIterationDiagnostics(10.0,9.0,1.05,0.857,1.05*0.857,100.0,SENTINEL_AI,20.0,21.0,1.05)
sentinel=IterativeThermalState(
    inside_bulk_temperature=500.0,outside_bulk_temperature=600.0,
    inside_wall_temperature=550.0,outside_wall_temperature=551.0,
    inside_bulk_props=sentinel_props,inside_wall_props=sentinel_props,
    outside_bulk_props=sentinel_props,outside_wall_props=sentinel_props,
    alfa_i=SENTINEL_AI,alfa_o=SENTINEL_AO,U=SENTINEL_U,UA=SENTINEL_UA,
    iterations=7,converged=True,residual=1e-3,diagnostics=sentinel_diag,
    inside_provider_name="SentinelProvider",outside_provider_name="SentinelProvider",warnings=(),
)
captured={}
def fake_solver(hx_arg,**kwargs):
    captured["hx"]=hx_arg; captured.update(kwargs); return sentinel

with mock.patch(patch_symbol,side_effect=fake_solver) as patched:
    sentinel_rating=hx.rate(inside_spec,outside_spec,effectiveness=0.30)

record_check("Sentinel solver called",patched.called,patched.call_count,">= 1")
record_check("Sentinel solver called exactly once",patched.call_count==1,patched.call_count,"1")
record_check("Sentinel received correct geometry",captured.get("hx") is hx,type(captured.get("hx")).__name__,"same exchanger object")
args_ok=(captured.get("m_dot_inside")==M_DOT_INSIDE and captured.get("m_dot_outside")==M_DOT_OUTSIDE and captured.get("inside_provider") is inside_provider and captured.get("outside_provider") is outside_provider and captured.get("T_in_inside")==T_COLD_IN and captured.get("T_in_outside")==T_HOT_IN)
record_check("Sentinel received flows, providers and inlet state",args_ok,{k:captured.get(k) for k in ("m_dot_inside","m_dot_outside","T_in_inside","T_in_outside")},"baseline closed-balance inputs")
sentinel_consumed=(sentinel_rating.alfa_i==SENTINEL_AI and sentinel_rating.alfa_o==SENTINEL_AO and sentinel_rating.U_mean==SENTINEL_U and sentinel_rating.UA_actual==SENTINEL_UA and sentinel_rating.thermal_state is sentinel)
record_check("Rating consumes sentinel values without legacy overwrite",sentinel_consumed,(sentinel_rating.alfa_i,sentinel_rating.alfa_o,sentinel_rating.U_mean,sentinel_rating.UA_actual),(SENTINEL_AI,SENTINEL_AO,SENTINEL_U,SENTINEL_UA))
record_check("Sentinel heat-duty argument",True,"solver API has no heat-duty parameter","verify if accepted","The actual solver derives its epsilon-NTU duty internally; geometry, flows, providers and inlet states were verified.",status="NOT APPLICABLE")
print({"patch_symbol":patch_symbol,"call_count":patched.call_count,"sentinel_values_consumed":sentinel_consumed,"captured_keys":sorted(captured)})


{'patch_symbol': 'core.models.rating.solve_iterative_thermal_state', 'call_count': 1, 'sentinel_values_consumed': True, 'captured_keys': ['T_in_inside', 'T_in_outside', 'euler_provider', 'flow_arrangement', 'hx', 'inside_provider', 'm_dot_inside', 'm_dot_outside', 'max_iterations', 'outside_provider', 'p_inside', 'p_outside', 'relative_alfa_tolerance', 'relaxation_factor', 'wall_temperature_tolerance_K']}


## 17. Final engineering summary


In [17]:
def group_status(prefixes):
    selected=[c for c in checks if any(p.lower() in c["Check"].lower() for p in prefixes)]
    if any(c["Status"]=="FAIL" for c in selected): return "FAIL"
    if selected and all(c["Status"]=="NOT APPLICABLE" for c in selected): return "NOT APPLICABLE"
    return "PASS"

engineering_summary=[
    ("Direct thermal iteration",group_status(["Direct iterative","Normal outside-hot"])),
    ("Physical wall ordering",group_status(["wall below","wall above","not colder","bracketed","Reverse wall"])),
    ("Bulk/wall property evaluation",group_status(["at reported temperature"])),
    ("Internal wall correction",group_status(["wall correction formula","heated-gas wall correction"])),
    ("Length correction",group_status(["Finite-length","Length correction"])),
    ("Outside wall correction",group_status(["outside correction"])),
    ("Nu-to-alfa consistency",group_status(["alfa from","final alfa_i","returned alfa"])),
    ("UA reconstruction",group_status(["UA reconstructed"])),
    ("Convergence handling",group_status(["iteration count","iteration limit","residual meets","Forced non-convergence"])),
    ("Reverse heat direction",group_status(["Reverse case","Reverse wall","Signed heat","Gas perspective"])),
    ("Rating integration",group_status(["Rating top-level","Rating exposes"])),
    ("Sentinel call-path test",group_status(["Sentinel solver","Sentinel received","consumes sentinel"])),
]
print_table([{"Engineering topic":k,"Status":v} for k,v in engineering_summary])
print("\nAll validation checks:")
print_table(checks)
all_warnings=[]
for label,state in (("outside-hot",outside_hot),("inside-hot",inside_hot),("forced non-convergence",forced),("rating",rating_result.thermal_state)):
    all_warnings.extend([{"case":label,**row} for row in warning_rows(state)])
print("\nWarnings encountered (duplicates preserved for audit):")
print_table(all_warnings)
counts={status:sum(c["Status"]==status for c in checks) for status in ("PASS","FAIL","NOT APPLICABLE")}
print("\nVALIDATION_COUNTS",counts)
print("SENTINEL_CONSUMED",sentinel_consumed)
print("FINITE_LENGTH_ACTIVE",outside_hot.diagnostics.inside_length_correction>1.0)
print("INTERNAL_WALL_FORMULA_MATCH",all(c["Status"]=="PASS" for c in checks if "wall correction formula" in c["Check"]))
print("UA_RECONSTRUCTABLE",all(c["Status"]=="PASS" for c in checks if "UA reconstructed" in c["Check"]))
failed=[check for check in checks if check["Status"]=="FAIL"]
assert not failed, f"{len(failed)} thermal-iteration validation checks failed"
print("FINAL_ASSERTION PASS")


Engineering topic             | Status
------------------------------+-------
Direct thermal iteration      | PASS  
Physical wall ordering        | PASS  
Bulk/wall property evaluation | PASS  
Internal wall correction      | PASS  
Length correction             | PASS  
Outside wall correction       | PASS  
Nu-to-alfa consistency        | PASS  
UA reconstruction             | PASS  
Convergence handling          | PASS  
Reverse heat direction        | PASS  
Rating integration            | PASS  
Sentinel call-path test       | PASS  

All validation checks:
Check                                                        | Status         | Calculated value                                                                                                                         | Expected condition                                   | Notes                                                                                                                
--------------------------------------

## Inlet, midpoint, and outlet fluid properties

Point states below come directly from the solver's existing hydraulic results. The wet midpoint uses arithmetic mean temperature and water ratio; no transport-property provider is called for presentation.

In [18]:
import math
import pandas as pd


def endpoint_property_table(solver_result, side):
    """Read solver-owned hydraulic point states without provider calls."""
    states = (
        ("inlet", getattr(solver_result, f"{side}_properties_inlet")),
        ("midpoint", getattr(solver_result, f"{side}_properties_midpoint")),
        ("outlet", getattr(solver_result, f"{side}_properties_outlet")),
    )
    return pd.DataFrame(
        [
            {
                "state": name,
                "T [°C]": state.T - 273.15,
                "p [Pa]": state.p,
                "rho [kg/m³]": state.rho,
                "cp [J/(kg·K)]": state.cp,
                "mu [Pa·s]": state.mu,
                "k [W/(m·K)]": state.k,
                "Pr [-]": state.Pr,
            }
            for name, state in states
            if state is not None
        ]
    ).set_index("state")


def representative_0d_property_table(solver_result):
    """Keep representative thermal properties separate from point states."""
    if hasattr(solver_result, "inside_props_mean"):
        pairs = (
            ("inside", solver_result.T_mean_inside, solver_result.inside_props_mean),
            ("outside", solver_result.T_mean_outside, solver_result.outside_props_mean),
        )
    elif getattr(solver_result, "thermal_state", None) is not None:
        thermal = solver_result.thermal_state
        pairs = (
            ("inside", thermal.inside_bulk_temperature, thermal.inside_bulk_props),
            ("outside", thermal.outside_bulk_temperature, thermal.outside_bulk_props),
        )
    else:
        return pd.DataFrame()
    return pd.DataFrame(
        [
            {
                "side": side,
                "T representative [°C]": temperature - 273.15,
                "rho [kg/m³]": props.rho,
                "cp [J/(kg·K)]": props.cp,
                "mu [Pa·s]": props.mu,
                "k [W/(m·K)]": props.k,
                "Pr [-]": props.mu * props.cp / props.k,
            }
            for side, temperature, props in pairs
        ]
    ).set_index("side")


def wet_gas_state_table(solver_result, outside_provider_for_result=None):
    """Combine hydraulic states with wet diagnostics already returned by the solver."""
    pc = getattr(solver_result, "outside_phase_change", None)
    if pc is None or not pc.capable or pc.W_in is None:
        return pd.DataFrame()

    W_mid = 0.5 * (pc.W_in + pc.W_out)
    dew_mid = math.nan
    if outside_provider_for_result is not None:
        from core.phase_change.capability import detect_phase_change_capability
        from core.phase_change.integration import _dew_point_at_ratio

        capability = detect_phase_change_capability(outside_provider_for_result)
        midpoint_state = solver_result.outside_properties_midpoint
        dew_mid_value = _dew_point_at_ratio(
            capability, W_mid, p=midpoint_state.p
        )
        dew_mid = math.nan if dew_mid_value is None else dew_mid_value

    dry_flow = pc.m_dot_dry_carrier
    vapor_mid = (
        math.nan
        if dry_flow is None
        else dry_flow * W_mid
    )
    gas_mid = (
        math.nan
        if dry_flow is None
        else dry_flow + vapor_mid
    )
    values = (
        ("inlet", pc.W_in, pc.dew_point_in, pc.m_dot_gas_in, pc.m_dot_water_vapor_in),
        ("midpoint", W_mid, dew_mid, gas_mid, vapor_mid),
        ("outlet", pc.W_out, pc.dew_point_out, pc.m_dot_gas_out, pc.m_dot_water_vapor_out),
    )
    return pd.DataFrame(
        [
            {
                "state": name,
                "W [kg/kg dry]": W,
                "dew point [°C]": (
                    math.nan if dew_point is None else dew_point - 273.15
                ),
                "m_dot gas [kg/s]": gas_flow,
                "m_dot water vapor [kg/s]": vapor_flow,
            }
            for name, W, dew_point, gas_flow, vapor_flow in values
        ]
    ).set_index("state")


def condensation_summary_table(solver_result):
    pc = getattr(solver_result, "outside_phase_change", None)
    if pc is None or not pc.capable:
        return pd.DataFrame()
    return pd.DataFrame(
        [
            {
                "m_dot condensate [kg/s]": pc.m_dot_condensate,
                "Q_sensible [W]": pc.Q_sensible,
                "Q_latent [W]": pc.Q_latent,
                "wet_surface_fraction [-]": pc.wet_surface_fraction,
                "wall Tmin [°C]": (
                    math.nan if pc.wall_temperature_min is None
                    else pc.wall_temperature_min - 273.15
                ),
                "wall Tmean [°C]": (
                    math.nan if pc.wall_temperature_mean is None
                    else pc.wall_temperature_mean - 273.15
                ),
                "wall Tmax [°C]": (
                    math.nan if pc.wall_temperature_max is None
                    else pc.wall_temperature_max - 273.15
                ),
            }
        ],
        index=["outside"],
    )

In [19]:
endpoint_results = [('Rating', rating_result)]
outside_provider_for_endpoint_table = outside_provider

for result_label, endpoint_result in endpoint_results:
    print(result_label)
    print("Inside")
    display(endpoint_property_table(endpoint_result, "inside"))
    print("Outside")
    display(endpoint_property_table(endpoint_result, "outside"))

    wet_table = wet_gas_state_table(
        endpoint_result, outside_provider_for_endpoint_table
    )
    if not wet_table.empty:
        print("Outside wet-gas mass and dew-point diagnostics")
        display(wet_table)
        display(condensation_summary_table(endpoint_result))

Rating
Inside


,T [°C],p [Pa],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
state,,,,,,,
inlet,6.850000,101325.0,1.261325,1005.806597,0.000018,0.024883,0.709802
midpoint,119.383898,101325.0,0.899107,1013.274324,0.000023,0.032948,0.699246
outlet,231.917797,101325.0,0.698659,1030.842027,0.000027,0.040261,0.698582


Outside


,T [°C],p [Pa],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
state,,,,,,,
inlet,826.850000,101325.0,0.320804,1158.818206,0.000046,0.072680,0.734249
midpoint,701.390594,101325.0,0.362094,1136.101437,0.000043,0.066383,0.728327
outlet,575.931187,101325.0,0.415586,1109.837991,0.000039,0.059870,0.720618


## Representative 0D properties used by the solver

These lumped thermal-model properties are retained separately; they are not substitutes for inlet or outlet states.

In [20]:
for result_label, endpoint_result in endpoint_results:
    representative_table = representative_0d_property_table(endpoint_result)
    if not representative_table.empty:
        print(result_label)
        display(representative_table)

Rating


,T representative [°C],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
side,,,,,,
inside,226.683488,0.705978,1029.837060,0.000027,0.039934,0.698445
outside,571.652901,0.417691,1108.883705,0.000039,0.059643,0.720326
